## Method 5: MiniRocket + MLP fully optimised **V5**

### imports 

In [ ]:
# Core scientific stack
import numpy as np
import pandas as pd

# ---- Time‑series transforms ---------------------------------------------
from sktime.transformations.panel.rocket import MiniRocket
# Try to grab multivariate; fall back to univariate
try:
    from sktime.transformations.panel.rocket import MiniRocketMultivariate
except ImportError:
    MiniRocketMultivariate = MiniRocket

# 84‑feature reducer (only in newer versions); if absent we set to None
try:
    from sktime.transformations.panel.rocket import MiniRocketFeatures
except ImportError:
    MiniRocketFeatures = None

# ---- Classic ML ----------------------------------------------------------
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold

# ---- Utility -------------------------------------------------------------
import joblib, warnings, os, random
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

### load dataset & basic schema setup

* loading our CSV 
* check the shape, names etc 
* selecting the target column 
* selecting interaction type and ligand names
* selecting all the numeric columns 

In [13]:
import glob, os, pandas as pd, numpy as np

DATA_DIR  = "Data/New/trained"                 # folder holding the three csvs
csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
assert csv_paths, f"No CSV files found under {DATA_DIR!r}"

frames = []
for path in csv_paths:
    df_tmp = pd.read_csv(path)

    # ── 1. Rename the three duplicate‑header columns by absolute position ──
    cols = list(df_tmp.columns)
    cols[0]  = "frame_idx"
    cols[-2] = "bond_type"     # target
    cols[-1] = "drug_name"     # to be ignored
    df_tmp.columns = cols

    # ── 2. If the first data row is a mini header ("bond_type", "drug_name"), drop it
    if (df_tmp.loc[0, "bond_type"] == "bond_type") or (df_tmp.loc[0, "drug_name"] == "drug_name"):
        df_tmp = df_tmp.iloc[1:].reset_index(drop=True)

    frames.append(df_tmp)

# ── 3. Concatenate all files vertically ───────────────────────────────────
df = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(csv_paths)} files → combined shape: {df.shape}")

# ── 4. Identify schema parts ───────────────────────────────────────────────
TARGET_COL         = "bond_type"
CATEGORICAL_STATIC = []                    # drug_name ignored per spec
fingerprint_cols   = [c for c in df.columns
                      if c not in ["frame_idx", "bond_type", "drug_name"]]

# Cast fingerprint features to float32 for MiniRocket speed
df[fingerprint_cols] = df[fingerprint_cols].astype("float32")

print("\nColumns:", df.columns.tolist()[:12], "…")
print(f"Fingerprint channels: {len(fingerprint_cols)}")
display(df.head(3))

Loaded 3 files → combined shape: (9249, 89)

Columns: ['frame_idx', 'ALA331.P', 'ALA423.P', 'ALA423.P.1', 'ALA479.P', 'ALA480.P', 'ALA77.P', 'ALA77.P.1', 'ALA81.P', 'ASN353.P', 'ASN353.P.1', 'ASP421.P'] …
Fingerprint channels: 86


,frame_idx,ALA331.P,ALA423.P,ALA423.P.1,ALA479.P,ALA480.P,ALA77.P,ALA77.P.1,ALA81.P,ASN353.P,...,ALA423.P.2,GLY153.P.1,GLY426.P.1,PHE76.P.3,SER149.P.1,SER149.P.2,SER321.P.1,SER422.P.1,SER422.P.2,VAL152.P.2
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Split data & reshape for mini-rocket 

* using stratified to ensure classes are equally represneted 
* Tensor re-shape, turning them into **( n_samples, n_channels, n_timepoints )**, for MiniRocketMultivariate 
* seperating the categorical features  

In [19]:
# Cell 3 ────────────────────────────────────────────────────────────────────
"""
Goals
─────
1. Stratify‑split the combined dataframe into 70 % train, 15 % valid, 15 % test.
2. Convert the fingerprint columns into a 3‑D tensor with shape
       (n_samples, n_channels, n_timepoints)
   and **pad** the singleton time axis so that `n_timepoints = 9`
   (MiniRocketMultivariate requires ≥ 9).
3. Encode the string labels ('inward'/'outward'/'occluded') into integers
   and save the mapping for later inference.
4. Nothing from 'drug_name' or 'frame_idx' is used for learning.
"""

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np, os

# ── 1. Stratified split ────────────────────────────────────────────────────
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df[TARGET_COL], random_state=SEED
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[TARGET_COL], random_state=SEED
)

print(f"Train: {train_df.shape},  Valid: {valid_df.shape},  Test: {test_df.shape}")

# ── 2. Helper: dataframe slice → (N, C, 1) tensor --------------------------
def df_to_rocket_tensor(sub_df, feature_cols):
    X = sub_df[feature_cols].to_numpy(dtype=np.float32)          # (N, C)
    return X[..., None]                                          # (N, C, 1)

X_train_fp = df_to_rocket_tensor(train_df, fingerprint_cols)
X_valid_fp = df_to_rocket_tensor(valid_df, fingerprint_cols)
X_test_fp  = df_to_rocket_tensor(test_df,  fingerprint_cols)

# ── 2b. Pad the singleton time axis to length 9 ----------------------------
PAD_LEN = 9   # MiniRocketMultivariate minimum

def pad_to_len9(X):
    n, c, t = X.shape                          # t == 1
    pad_width = ((0, 0), (0, 0), (0, PAD_LEN - t))
    return np.pad(X, pad_width, mode="constant")

X_train_fp = pad_to_len9(X_train_fp)
X_valid_fp = pad_to_len9(X_valid_fp)
X_test_fp  = pad_to_len9(X_test_fp)

print("Tensor shapes after padding:", X_train_fp.shape,
                                      X_valid_fp.shape,
                                      X_test_fp.shape)

# ── 3. Encode labels -------------------------------------------------------
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df[TARGET_COL])
y_valid = label_encoder.transform(valid_df[TARGET_COL])
y_test  = label_encoder.transform(test_df[TARGET_COL])

# ── 4. Save label mapping --------------------------------------------------
os.makedirs("models/v5", exist_ok=True)
np.save("models/v5/label_encoder_classes.npy", label_encoder.classes_)
print("✅ Saved label mapping -> models/v5/label_encoder_classes.npy")
print("Classes:", list(label_encoder.classes_))


Train: (6474, 89),  Valid: (1387, 89),  Test: (1388, 89)
Tensor shapes after padding: (6474, 86, 9) (1387, 86, 9) (1388, 86, 9)
✅ Saved label mapping -> models/v5/label_encoder_classes.npy
Classes: ['inward', 'occluded', 'outward']


### model 

In [22]:
# Cell 4 ────────────────────────────────────────────────────────────────────
"""
Pipeline
1. Fit a MiniRocketMultivariate transformer on the *training* fingerprints.
2. Transform train/valid/test into fixed‑length feature vectors.
3. Feed those vectors to a 2‑hidden‑layer MLP in PyTorch.
4. Early‑stop on validation accuracy.
5. Save the fitted Rocket transformer and trained MLP weights to models/v5/.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
import joblib, os, numpy as np

# ── 1. Fit MiniRocket transformer ─────────────────────────────────────────
rocket = MiniRocketMultivariate(random_state=SEED)  # default 9 ,996 features
rocket.fit(X_train_fp)

X_train_vec = rocket.transform(X_train_fp)   # (N_train, 9 ,996)
X_valid_vec = rocket.transform(X_valid_fp)
X_test_vec  = rocket.transform(X_test_fp)

n_features = X_train_vec.shape[1]
print("Rocket feature dim:", n_features)

# ── 2. Torch tensors -------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def to_tensor(arr):
    """Ensure ndarray→float32→torch tensor on correct device."""
    if hasattr(arr, "values"):               # handles pandas DataFrame
        arr = arr.values
    arr = np.asarray(arr, dtype=np.float32)  # dense contiguous array
    return torch.tensor(arr, device=device)

train_X = to_tensor(X_train_vec)
valid_X = to_tensor(X_valid_vec)
test_X  = to_tensor(X_test_vec)

train_y = torch.tensor(y_train, dtype=torch.long, device=device)
valid_y = torch.tensor(y_valid, dtype=torch.long, device=device)
test_y  = torch.tensor(y_test,  dtype=torch.long, device=device)

# ── 3. Define the MLP ------------------------------------------------------
class RocketMLP(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )
    def forward(self, x):
        return self.net(x)

model = RocketMLP(in_dim=n_features, n_classes=len(label_encoder.classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ── 4. Training loop with early stopping ----------------------------------
EPOCHS, patience = 50, 5
best_val, patience_cnt = 0.0, 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    # ---- train
    model.train()
    optimizer.zero_grad()
    logits = model(train_X)
    loss = criterion(logits, train_y)
    loss.backward()
    optimizer.step()

    # ---- validate
    model.eval()
    with torch.no_grad():
        val_logits = model(valid_X)
        val_pred = val_logits.argmax(1)
        val_acc = accuracy_score(valid_y.cpu(), val_pred.cpu())

    print(f"Epoch {epoch:02d} | train_loss {loss.item():.4f} | val_acc {val_acc:.4f}")

    # early stopping
    if val_acc > best_val:
        best_val = val_acc
        patience_cnt = 0
        best_state = model.state_dict()
    else:
        patience_cnt += 1
        if patience_cnt >= patience:
            print("Early stopping")
            break

# load best weights
model.load_state_dict(best_state)

# ── 5. Final test accuracy -------------------------------------------------
model.eval()
with torch.no_grad():
    test_pred = model(test_X).argmax(1)
    test_acc  = accuracy_score(test_y.cpu(), test_pred.cpu())
print(f"\n✅ Test accuracy: {test_acc:.4f}")

# ── 6. Persist artefacts ---------------------------------------------------
os.makedirs("models/v5", exist_ok=True)
joblib.dump(rocket,  "models/v5/minirocket_transformer.pkl")
np.save("models/v5/minirocket_input_dim.npy", [n_features])
torch.save(model.state_dict(), "models/v5/rocket_mlp_weights.pt")
print("Saved transformer and MLP weights to models/v5/")


Rocket feature dim: 9996
Epoch 01 | train_loss 1.1010 | val_acc 0.8327
Epoch 02 | train_loss 0.8021 | val_acc 0.9950
Epoch 03 | train_loss 0.4183 | val_acc 1.0000
Epoch 04 | train_loss 0.1823 | val_acc 1.0000
Epoch 05 | train_loss 0.0719 | val_acc 1.0000
Epoch 06 | train_loss 0.0267 | val_acc 1.0000
Epoch 07 | train_loss 0.0107 | val_acc 1.0000
Epoch 08 | train_loss 0.0046 | val_acc 1.0000
Early stopping

✅ Test accuracy: 1.0000
Saved transformer and MLP weights to models/v5/
